In [1]:
# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================

# !pip install -q \
# langgraph \
# langchain \
# langchain-community \
# langchain-huggingface \
# langchain-openai \
# pypdf \
# faiss-cpu \
# sentence-transformers

In [2]:
# !pip install -q langchain-openai

In [3]:
# ============================================================
# IMPORTS
# ============================================================

from typing import TypedDict, List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [9]:
# ============================================================
# CONFIGURAÇÃO DO OPENAI
# ============================================================
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [10]:
# ============================================================
# CONFIGURAÇÃO DO EMBEDDING
# ============================================================

# Modelo excelente para RAG médico e português
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [11]:
# ============================================================
# CONFIGURAÇÃO DA BASE DE CONHECIMENTO (RAG)
# ============================================================

def configurar_base_conhecimento(caminhos_pdfs: List[str]):

    documentos_totais = []

    for caminho in caminhos_pdfs:

        loader = PyPDFLoader(caminho)

        documentos = loader.load()

        # Adicionando metadata importante
        for doc in documentos:
            doc.metadata["fonte"] = caminho
            doc.metadata["tipo"] = "documento_medico"

        documentos_totais.extend(documentos)

    # Melhor chunking para contexto clínico
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=300
    )

    docs = splitter.split_documents(documentos_totais)

    # Banco vetorial
    vectorstore = FAISS.from_documents(
        docs,
        embedding_model
    )

    # Retriever melhorado com MMR
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 5,
            "fetch_k": 20
        }
    )

    return retriever

In [12]:
# ============================================================
# CARREGAMENTO DOS PDFs
# ============================================================

retriever = configurar_base_conhecimento([
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/faqs.pdf",
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/protocolos.pdf",
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/receitas.pdf",
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/triagens.pdf",
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/laudos.pdf",
    "/content/drive/MyDrive/Tech-Challenge-FAse-3/arquivos-assistente/evolucoes.pdf"
])



In [13]:
# ============================================================
# ESTADO GLOBAL DO GRAFO
# ============================================================

class MedicalState(TypedDict):

    pergunta: str

    contexto_recuperado: str

    sugestao_conduta: str

    validado_por_humano: bool

    historico: List[str]


In [14]:
# ============================================================
# AGENTE PESQUISADOR (RAG)
# ============================================================

def agente_pesquisador(state: MedicalState):

    pergunta = state["pergunta"]

    docs = retriever.invoke(pergunta)

    contexto = "\n\n".join([doc.page_content for doc in docs])

    fontes = "\n".join([
        f"- {doc.metadata.get('fonte', 'desconhecida')}"
        for doc in docs
    ])

    contexto_final = f"""
CONTEXTO CLÍNICO:

{contexto}

FONTES:
{fontes}
"""

    return {
        "contexto_recuperado": contexto_final
    }


In [15]:


# ============================================================
# AGENTE ANALISTA CLÍNICO
# ============================================================

def agente_analista(state: MedicalState):

    llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
    )

    prompt = f"""
Você é um assistente clínico hospitalar especializado.

REGRAS IMPORTANTES:
- Responda SOMENTE com base no contexto fornecido.
- Nunca invente protocolos.
- Nunca invente medicamentos.
- Nunca gere diagnósticos definitivos.
- Se não houver informação suficiente, diga isso claramente.
- Cite os protocolos e fontes quando possível.
- Sua resposta NÃO substitui avaliação médica humana.
- Seja técnico, claro e objetivo.

CONTEXTO:
{state['contexto_recuperado']}

PERGUNTA:
{state['pergunta']}
"""

    resposta = llm.invoke(prompt)

    return {
        "sugestao_conduta": resposta.content
    }


In [16]:
# ============================================================
# VALIDAÇÃO HUMANA (HITL)
# ============================================================

def validar_sugestao(state: MedicalState):

    print("\n")
    print("=" * 60)
    print("REVISÃO MÉDICA NECESSÁRIA")
    print("=" * 60)

    print("\nPERGUNTA:")
    print(state["pergunta"])

    print("\nSUGESTÃO GERADA:")
    print(state["sugestao_conduta"])

    print("\n")

    confirmacao = input("Aprovar resposta? (s/n): ")

    aprovado = confirmacao.lower() == "s"

    return {
        "validado_por_humano": aprovado
    }

In [17]:
# ============================================================
# DEFINIÇÃO DO GRAFO
# ============================================================

workflow = StateGraph(MedicalState)


# ============================================================
# NÓS
# ============================================================

workflow.add_node(
    "pesquisador",
    agente_pesquisador
)

workflow.add_node(
    "analista",
    agente_analista
)

workflow.add_node(
    "validador",
    validar_sugestao
)

In [18]:
# ============================================================
# FLUXO
# ============================================================

workflow.set_entry_point("pesquisador")

workflow.add_edge(
    "pesquisador",
    "analista"
)

workflow.add_edge(
    "analista",
    "validador"
)



In [19]:

# ============================================================
# ROTA CONDICIONAL
# ============================================================

def rota_pos_validacao(state: MedicalState):

    if state["validado_por_humano"]:
        return "fim"

    return "analista"


workflow.add_conditional_edges(
    "validador",
    rota_pos_validacao,
    {
        "fim": END,
        "analista": "analista"
    }
)

In [20]:

# ============================================================
# CHECKPOINT / MEMÓRIA
# ============================================================

memoria = MemorySaver()

app = workflow.compile(
    checkpointer=memoria
)


In [21]:

# ============================================================
# EXECUÇÃO
# ============================================================

config = {
    "configurable": {
        "thread_id": "consulta_001"
    }
}

entrada = {
    "pergunta": "Qual o protocolo para dor torácica aguda?",
    "historico": []
}



In [22]:
# ============================================================
# STREAM DO GRAFO
# ============================================================

for output in app.stream(entrada, config):

    for key, value in output.items():

        print("\n")
        print(f"Nó executado: {key}")


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 60)
print("PROCESSO FINALIZADO")
print("=" * 60)



Nó executado: pesquisador


Nó executado: analista


REVISÃO MÉDICA NECESSÁRIA

PERGUNTA:
Qual o protocolo para dor torácica aguda?

SUGESTÃO GERADA:
Não há informação suficiente sobre um protocolo específico para dor torácica aguda no contexto fornecido. Recomendo consultar um profissional de saúde para uma avaliação adequada.


Aprovar resposta? (s/n): n


Nó executado: validador


Nó executado: analista


REVISÃO MÉDICA NECESSÁRIA

PERGUNTA:
Qual o protocolo para dor torácica aguda?

SUGESTÃO GERADA:
Não há informação suficiente no contexto fornecido sobre um protocolo específico para dor torácica aguda. Recomendo consultar um profissional de saúde ou as diretrizes clínicas apropriadas para essa condição.


Aprovar resposta? (s/n): n


Nó executado: validador


Nó executado: analista


REVISÃO MÉDICA NECESSÁRIA

PERGUNTA:
Qual o protocolo para dor torácica aguda?

SUGESTÃO GERADA:
Não há informação suficiente no contexto fornecido sobre um protocolo específico para dor torácica ag